# 07 · Illusion mechanism (rebuttal analyses)
Reproduces: decoding errors at the counterfactual checkpoints (Fig. 2f tests), the Hassenstein–Reichardt 2-point correlation per condition, the closeness of the T4/T5 code to reverse-φ, and the texture optimization of the 2-point correlation. Set `GENERATE = True` to regenerate the data on a GPU (hours); the analyses only read the saved files.

In [ ]:
import json, os
import numpy as np
from scipy import stats
from flynet_robustness.analysis import illusion as il
DATA = os.environ.get('FLYNET_ILLUSION_DATA', 'data/illusion')
TRAJ, SIMS, OPT = (f'{DATA}/{d}' for d in ('trajectories_fooling', 'simulations_ring', 'optimized_textures_correlations'))
TRAJECTORIES = json.load(open('data/configs/illusion_trajectories.json'))   # 39 hit trajectories FlyNet detects
GENERATE = False

## 1. Data (GPU)
Per trajectory: counterfactual restoration (illusion at step 250, restored at 500), 2-point texture optimization (Δ = 8 frames, 300 steps, minimize / maximize), then the four condition simulations (original, illusion, recovered, reverse-φ flipping contrast every 33 frames).

In [ ]:
if GENERATE:
    from flynet_robustness import config
    from flynet_robustness.attacks import counterfactual_restoration, run_correlation_optimization
    from flynet_robustness.dataset import save_condition_simulations
    from flynet_robustness.decoders import load_decoder, load_decoder_pos_vel
    from flynet_robustness.models import get_flynet
    model1, _, idxs1, _ = get_flynet('FlyNet')
    net, decoder = (model1, idxs1, None, None), load_decoder('FlyNet')
    pos_vel = (load_decoder_pos_vel('FlyNet', load_velocity=False), load_decoder_pos_vel('FlyNet'))
    for t in TRAJECTORIES:
        kw = dict(config.DEFAULT_STIMULUS, radius=t['radius'], theta=t['theta'], z0=t['z0'], time=t['time'],
                  device='cuda', a=1)
        folder = f"optimal_trajectories_fooling_{t['id']}"
        counterfactual_restoration(net, decoder, texttype=t['texture'], base_kwargs=kw, device='cuda',
                                   out_dir=f'{TRAJ}/{folder}', pos_vel=pos_vel)
        for d in ('minimize', 'maximize'):
            run_correlation_optimization(net, decoder, maximize=d == 'maximize', texttype=t['texture'], base_kwargs=kw,
                                         out_dir=f'{OPT}/{folder}/{d}', pos_vel=pos_vel, device='cuda')
    save_condition_simulations(model1, idxs1, TRAJ, SIMS)

## 2. Decoding errors at the counterfactual checkpoints (Fig. 2f)
Paired t-tests over the first 30 trajectories.

In [ ]:
def history(root, t, sub=''):
    return json.load(open(f"{root}/optimal_trajectories_fooling_{t['id']}/{sub}{t['texture']}_loss_history.json"))

H = [{e['step']: e for e in history(TRAJ, t)} for t in TRAJECTORIES[:30]]
for key in ('mse_pos', 'mse_vel'):
    e = np.array([[h[s][key] for s in (1, 250, 500)] for h in H])          # initial / illusion / restored
    p = [stats.ttest_rel(e[:, i], e[:, j]).pvalue for i, j in ((0, 1), (1, 2), (0, 2))]
    print(f'{key}: mean {e.mean(0).round(5)} | p initial-illusion {p[0]:.1e}, illusion-restored {p[1]:.1e}, '
          f'initial-restored {p[2]:.1e}')

## 3. Hassenstein–Reichardt 2-point correlation, Δ = 8 frames (80 ms)
Each hex vs its next-outward radial neighbour, contrast relative to each frame's mean, averaged over hexes, frame pairs and trajectories. Reverse-φ: only frame pairs spanning a contrast flip.

In [ ]:
N = il.trials(SIMS)[:30]
g = {c: np.array([il.two_point(il.load_stimulus(SIMS, n, c), 8) for n in N]) for c in ('original', 'illusion', 'recovered')}
g['reverse_phi'] = np.array([il.two_point(s, 8, il.flip_mask(len(s), 8, 33))
                             for s in (il.load_stimulus(SIMS, n, 'reverse_phi') for n in N)])
for c, v in g.items():
    print(f'{c:12s} {v.mean():+.4f} (SEM {v.std() / np.sqrt(len(v)):.4f})')
p = lambda a, b: stats.ttest_rel(g[a], g[b]).pvalue
print(f"p original-illusion {p('original', 'illusion'):.1e} | illusion-recovered {p('illusion', 'recovered'):.1e} | "
      f"illusion-reverse_phi {p('illusion', 'reverse_phi'):.2f}")

## 4. Closeness of the T4/T5 code to reverse-φ
All 5,768 T4/T5 units, mean activity in the 200 ms after the first reverse-φ switch (same frames in every condition). Projection onto the original→reverse-φ axis (0 = original, 1 = reverse-φ). Paired t-tests.

In [ ]:
P = {c: np.stack([il.switch_locked(il.load_response(SIMS, n, c)) for n in N]) for c in il.CONDITIONS}
for metric, v in il.closeness(P).items():
    p = [stats.ttest_rel(v['illusion'], v[c]).pvalue for c in ('original', 'recovered')]
    print(f"{metric:10s} illusion {v['illusion'].mean():.3f}  recovered {v['recovered'].mean():.3f}  "
          f"original {v['original'].mean():.3f} | p vs original {p[0]:.1e}, vs recovered {p[1]:.1e}")

## 5. Texture optimization of the 2-point correlation
The network is not in the loss; p(hit) is only read out (Black and White trajectories).

In [ ]:
for d in ('minimize', 'maximize'):
    H = [history(OPT, t, f'{d}/') for t in TRAJECTORIES if t['texture'] in ('Black', 'White')]
    ph = np.array([[h[0]['p_hit'], h[-1]['p_hit']] for h in H])
    corr = np.array([[h[0]['correlation'], h[-1]['correlation']] for h in H])
    print(f'{d}: n={len(H)}  p(hit) {ph[:, 0].mean():.2f} -> {ph[:, 1].mean():.2f}   '
          f'2-point {corr[:, 0].mean():+.4f} -> {corr[:, 1].mean():+.4f}')